# Pump.fun Trading Strategy — Full Analysis Pipeline

**Complete circle:** download replay data → run trading engine → export → XGBoost/SHAP/statistical analysis → Optuna weight optimization → report.

Run cells in order. No GPU required — runs on free Colab CPU.

In [ ]:
# Cell 1: Install dependencies (~2 min)
!pip install -q zstandard pandas numpy scipy scikit-learn xgboost shap optuna matplotlib plotly statsmodels
print("Dependencies installed.")

In [ ]:
# Cell 2: Upload the analysis scripts or clone repo
import os, sys, urllib.request, zipfile, io

# Option A: If you uploaded scripts/analysis/ to Colab, skip this cell
# Option B: Clone from GitHub (replace with your repo URL)
# !git clone https://github.com/YOUR_USER/YOUR_REPO.git
# %cd YOUR_REPO/scripts/analysis

# Option C: Upload from local machine (recommended for first run)
from google.colab import files
print("Upload the analysis folder as a zip, or upload individual files.")
print("See the README for folder structure.")

# Check if we have the modules
try:
    import config
    from runner import ReplayRunner
    from analysis import run_analysis, print_report, optuna_optimize
    from analysis import load_trades_from_csv, avail_features
    from analysis import xgboost_regression, shap_analysis
    from pipeline import export_to_csv
    print("Modules loaded successfully.")
except ImportError as e:
    print(f"Import error: {e}")
    print("Modules not found. Upload the analysis folder or provide a path.")

In [ ]:
# Cell 3: Build and run the trading engine on replay data
import time

HOURS = 2  # change for more/fewer replay hours

runner = ReplayRunner(sol_balance=10.0, sol_amount=0.01)
runner.download_and_replay_recent(HOURS)
runner.print_summary()

result = runner.get_result()
trades = result['trades']
rejected = result['rejected']
print(f"\nTrades: {len(trades)}  Rejected signals: {len(rejected)}")

In [ ]:
# Cell 4: Export trades and rejected signals to CSV
paths = export_to_csv(trades, rejected)
print(f"Trades CSV: {paths.get('trades', 'N/A')}")
print(f"Rejected CSV: {paths.get('rejected', 'N/A')}")

# Show summary table
import pandas as pd
if 'trades' in paths:
    df = pd.read_csv(paths['trades'])
    print(f"\n{len(df)} trades columns:\n{list(df.columns)}")
    display(df.head())

In [ ]:
# Cell 5: Full analysis — XGBoost PnL regression + SHAP + feature importance
df = load_trades_from_csv(paths['trades'])

from analysis import avail_features, xgboost_regression, shap_analysis
if len(avail_features(df)) >= 2:
    result = run_analysis(paths['trades'], paths.get('rejected'))
    print_report(result)
else:
    print("Not enough trades with feature data for ML analysis.")

In [ ]:
# Cell 6: SHAP per-trade explanation (shows WHY each trade scored what it did)
import shap
import matplotlib.pyplot as plt
import numpy as np

if 'result' in locals() and result.get('xgb') and result.get('shap'):
    feature_names = result['xgb']['feature_names']
    shap_result = result['shap']
    shap_values = shap_result['shap_values']

    # SHAP beeswarm — shows how each feature pushes PnL prediction up/down
    plt.figure(figsize=(10, 6))
    shap.plots.beeswarm(shap_values, show=False)
    plt.title('SHAP Feature Impact on Predicted PnL')
    plt.tight_layout()
    plt.show()

    # SHAP bar — global importance
    plt.figure(figsize=(8, 4))
    shap.plots.bar(shap_values, show=False)
    plt.title('Mean |SHAP| — Global Feature Importance (PnL Regression)')
    plt.tight_layout()
    plt.show()

    # Waterfall for first test trade
    shap.plots.waterfall(shap_values[0], show=False)
    plt.title('SHAP Waterfall — Trade #0 PnL Explanation')
    plt.tight_layout()
    plt.show()
else:
    print("Run Cell 5 first to generate XGBoost results.")

In [ ]:
# Cell 7: Optuna full-parameter optimization (walk-forward validated)
N_TRIALS = 1000

opt_result = optuna_optimize(df, n_trials=N_TRIALS)
if opt_result:
    print(f"Train PF (in-sample):  {opt_result['train_pf']:.2f} ({opt_result['n_train']} trades)")
    print(f"Test PF (out-of-sample): {opt_result['test_pf']:.2f} ({opt_result['n_test']} trades)")
    print(f"\nCurrent v4 weights:")
    print(f"  wallet=0.40  age=0.45  liq=0.15")
    print(f"\nOptuna optimal (normalized):")
    b = opt_result['best_weights_norm']
    print(f"  wallet={b['wallet']:.3f}  age={b['age']:.3f}  liq={b['liq']:.3f}")
    print(f"\nAll optimal parameters:")
    for k, v in sorted(opt_result['best_params'].items()):
        print(f"  {k}: {v}")
else:
    print("Not enough data for Optuna optimization (need 10+ trades).")

In [ ]:
# Cell 8: Bucket analysis — wallet count, signal age, buy ratio
from analysis import bucket_analysis

buckets = bucket_analysis(df)

for key, title in [('score_buckets', 'Score Buckets'),
                    ('wallet_buckets', 'Wallet Buckets'),
                    ('age_buckets', 'Signal Age Buckets'),
                    ('exit_reasons', 'Exit Reasons')]:
    bdf = buckets.get(key)
    if bdf is not None and len(bdf):
        print(f"\n{title}:")
        display(bdf)

In [ ]:
# Cell 9: Equity curve + rolling metrics (PF, WR, DD)
import matplotlib.pyplot as plt

if len(trades) > 1:
    sorted_trades = sorted(trades, key=lambda t: t.exit_time)
    cum_pnl = []
    running = 0
    for t in sorted_trades:
        running += t.pnl
        cum_pnl.append(running)

    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

    # Equity curve
    axes[0].plot(cum_pnl, marker='.', linestyle='-', linewidth=1, color='blue')
    axes[0].axhline(0, color='gray', linestyle='--', alpha=0.5)
    axes[0].set_ylabel('Cumulative PnL ($)')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_title('Equity Curve')

    # Rolling PF and WR
    roll = result.get('rolling')
    if roll is not None and not roll.empty:
        ax2 = axes[1]
        ax2.plot(roll['rolling_pf'], label='Rolling PF', color='green', linewidth=1)
        ax2.axhline(result['pf'], color='green', linestyle='--', alpha=0.5, label=f'Overall PF={result["pf"]:.2f}')
        ax2.set_ylabel('Profit Factor')
        ax2.legend(loc='upper left')
        ax2.grid(True, alpha=0.3)

        ax2b = axes[1].twinx()
        ax2b.plot(roll['rolling_wr'], label='Rolling WR', color='orange', linewidth=1)
        ax2b.set_ylabel('Win Rate')
        ax2b.legend(loc='upper right')

        axes[1].set_title('Rolling PF & WR (20-trade window)')

        # Drawdown (from cumulative realized equity)
        axes[2].fill_between(range(len(roll)), 0, roll['drawdown'] * 100, color='red', alpha=0.3)
        axes[2].plot(roll['drawdown'] * 100, color='red', linewidth=1)
        axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
        axes[2].set_ylabel('Drawdown (%)')
        axes[2].set_xlabel('Trade # (rolling window)')
        axes[2].grid(True, alpha=0.3)
        axes[2].set_title(f'Drawdown (max: {roll["drawdown"].min()*100:.1f}%)')

    plt.tight_layout()
    plt.show()
else:
    print("Not enough trades for charts.")

In [ ]:
# Cell 10: Download CSVs for local analysis
from google.colab import files
import os

csv_dir = '/content/'
if 'paths' in locals():
    for p in paths.values():
        if os.path.exists(p):
            files.download(p)
            print(f"Downloaded: {p}")

print("\nDone! All analysis files are available in the Colab file browser.")